In [ ]:
import pandas as pd

# **Парсинг данных по форме 0409102 ЦБ РФ**

In [ ]:
def extract_profit(regn: int, quarter: int, year: int):
    if (quarter < 4 and year == 2013) or year < 2013:
        return None
    if year >= 2016:
        profit_code = 81201
        loss_code = 81202
    elif year in range(2013, 2016):
        profit_code = 31001
        loss_code = 31002
    else:
        return None
    if year == 2022 or (year == 2023 and quarter == 1):
        return None
    if year >= 2025:
        if quarter == 4:
            month_str = str(quarter * 3)
        else:
            month_str = "0" + str(quarter * 3)
        filename = month_str + str(year) + "_P1.csv"
    else:
        filename = str(quarter) + str(year) + "_P1.csv"
    df = pd.read_csv("https://github.com/JarlPenguin/macroeconomic_analysis_2026/raw/refs/heads/main/csv/" + filename)
    regn_mask = df["REGN,N,4,0"] == regn
    profit_mask = df["CODE,C,5"] == profit_code
    loss_mask = df["CODE,C,5"] == loss_code
    profit = df[regn_mask & profit_mask]["SIM_ITOGO,N,16,0"].item()
    loss = df[regn_mask & loss_mask]["SIM_ITOGO,N,16,0"].item()
    return profit if profit > 0 else -1 * loss

Проверка работы функции extract_profit

In [ ]:
extract_profit(1481, 1, 2025)

442918009.0

In [ ]:
for year in range(2013, 2026):
    for quarter in range(1, 5):
        if year == 2013 and quarter < 4:
            continue
        print(extract_profit(1481, quarter, year))

392634997
99348658
186415816
286692267
305703229
26319633
81583122
144432688
236256123
140983555
295130943
444822578
576834469
153400924
312069543
506446440
684529581.0
204724848.0
364286221
543875129.0
735364472
230796093.0
499678983.0
742353529.0
981369979.0
160426121.0
388987872.0
566608233.0
815114038.0
190878973.0
510621870.0
821684461.0
1046256855.0
None
None
None
None
None
672295598.0
899281222.0
1327142566.0
247116075.0
445416341.0
740552475.0
1258576344.0
442918009.0
1014706224.0
1461422848.0
1932024367.0


# **Парсинг данных по форме 0409101 ЦБ**

In [ ]:
def extract_101_metric(regn: int, quarter: int, year: int):
    if (quarter < 4 and year == 2013) or year < 2013:
        return None
    if year == 2022 or (year == 2023 and quarter in range(1, 3)):
        return None
    month_strs = [str(i) if len(str(i)) > 1 else "0" + str(i) for i in range(quarter * 3 - 3, quarter * 3 + 1)]
    for i in range(len(month_strs)):
        if int(month_strs[i]) < 1:
            month_strs[i] = str(int(month_strs[i]) + 12)
    year_files = [str(year - 1) if month_strs[0] == "12" else str(year)]
    year_files.extend([str(year) for i in range(3)])
    avg_active_sum = 0
    for month_str, year_file, i in zip(month_strs, year_files, range(4)):
        filename = month_str + year_file + "B1.csv"
        df = pd.read_csv("https://github.com/JarlPenguin/macroeconomic_analysis_2026/raw/refs/heads/main/csv/" + filename)
        regn_mask = df["REGN,N,4,0"] == regn
        plan_mask = df["PLAN,C,1"] == "А"
        num_sc_mask = df["NUM_SC,C,5"] == "ITGAP"
        active_mask = df["A_P,C,1"] == 1
        vitg_column = next(c for c in df.columns if c.startswith("VITG"))
        active_sum = df[regn_mask & plan_mask & num_sc_mask & active_mask][vitg_column].item()
        if i == 0 or i == 3:
            avg_active_sum += 0.5 * active_sum
        else:
            avg_active_sum += active_sum
    avg_active_sum /= 3

    return avg_active_sum

Проверка работы extract_101_metric

In [ ]:
extract_101_metric(1481, 4, 2025)

606056371276.6666

# **Формирование панельного датасета с учетом метаданных по макроиндикаторам**

In [ ]:
macro_df = pd.read_csv('metadata.csv')

macro_df.columns = [col.strip() for col in macro_df.columns]

bank_ids = [1481, 1000, 354, 2673, 3292] #id банков для дальнейшего анализа
panel_data = []

for regn in bank_ids:
    for year in range(2013, 2026):
        for quarter in range(1, 5):
            if year == 2013 and quarter < 3:
                continue
            if year == 2026:
                break

            profit = extract_profit(regn, quarter, year)
            active_sum = extract_101_metric(regn, quarter, year)

            date_str = f"{year}-Q{quarter}"

            panel_data.append({
                'bank_id': regn,
                'Date': date_str,
                'net_profit': profit,
                'avg_active_sum': active_sum
            })
banks_df = pd.DataFrame(panel_data)

final_panel = pd.merge(banks_df, macro_df, on='Date', how='left')

final_panel['roa'] = final_panel['net_profit'] / final_panel['avg_active_sum']

final_panel.set_index(['bank_id', 'Date'], inplace=True)

final_panel = final_panel.dropna(subset=['net_profit', 'avg_active_sum'])
display(final_panel.head(10))

net_profit  avg_active_sum  Average Key Rate  \
bank_id Date                                                     
1481    2013-Q4  392634997.0    3.929550e+10              5.50   
        2014-Q1   99348658.0    4.114557e+10              6.00   
        2014-Q2  186415816.0    4.140066e+10               NaN   
        2014-Q3  286692267.0    4.952186e+10              8.00   
        2014-Q4  305703229.0    6.328272e+10             10.33   
        2015-Q1   26319633.0    1.029691e+11             15.33   
        2015-Q2   81583122.0    9.353034e+10             13.17   
        2015-Q3  144432688.0    1.116698e+11             11.17   
        2015-Q4  236256123.0    1.457843e+11             11.00   
        2016-Q1  140983555.0    1.811348e+11             11.00   

                 Average Inflation Average GDP  Average USD/RUB Rate       roa  
bank_id Date                                                                    
1481    2013-Q4               6.41   20 104.35                 32.54  0.009992  
        2014-Q1               6.40   17 311.39                 35.14  0.002415  
        2014-Q2                NaN         NaN                   NaN  0.004503  
        2014-Q3               7.68   20 544.00                 36.16  0.005789  
        2014-Q4               9.38   22 223.32                 47.50  0.004831  
        2015-Q1              16.20   19 308.23                 62.19  0.000256  
        2015-Q2              15.82  20 380.08                  52.65  0.000872  
        2015-Q3              15.68   22 097.35                 63.02  0.001293  
        2015-Q4              14.37  24 345.02                  65.94  0.001621  
        2016-Q1               8.35   20 285.52                 74.63  0.000778

In [ ]:
final_panel

net_profit  avg_active_sum  Average Key Rate  \
bank_id Date                                                     
1481    2013-Q4  392634997.0    3.929550e+10              5.50   
        2014-Q1   99348658.0    4.114557e+10              6.00   
        2014-Q2  186415816.0    4.140066e+10               NaN   
        2014-Q3  286692267.0    4.952186e+10              8.00   
        2014-Q4  305703229.0    6.328272e+10             10.33   
...                      ...             ...               ...   
3292    2024-Q4  137371593.0    1.533151e+10             21.00   
        2025-Q1    7697980.0    1.852105e+10             21.00   
        2025-Q2   95956966.0    1.388870e+10             20.67   
        2025-Q3  123716688.0    1.432968e+10             17.67   
        2025-Q4   89491255.0    1.682241e+10             16.33   

                 Average Inflation Average GDP  Average USD/RUB Rate       roa  
bank_id Date                                                                    
1481    2013-Q4               6.41   20 104.35                 32.54  0.009992  
        2014-Q1               6.40   17 311.39                 35.14  0.002415  
        2014-Q2                NaN         NaN                   NaN  0.004503  
        2014-Q3               7.68   20 544.00                 36.16  0.005789  
        2014-Q4               9.38   22 223.32                 47.50  0.004831  
...                            ...         ...                   ...       ...  
3292    2024-Q4               8.98   59 271.00                 99.61  0.008960  
        2025-Q1              10.11   47 950.38                 92.42  0.000416  
        2025-Q2               9.84   49 284.77                 80.94  0.006909  
        2025-Q3               8.30   54 671.82                 80.56  0.008634  
        2025-Q4               6.65  62 354.06                  79.89  0.005320  

[215 rows x 7 columns]